In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

# ==========================================
# 1. LOAD DATASET KOMDIGI
# ==========================================
data_candidates = [
    Path("data/tabular/komdigi_hoaks.csv"),
    Path("../../../data/tabular/komdigi_hoaks.csv"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("File data/tabular/komdigi_hoaks.csv tidak ditemukan.")

data = pd.read_csv(DATA_PATH, usecols=["title", "body_text", "topics"])
data["text"] = data["title"].fillna("") + " " + data["body_text"].fillna("")
data["topics"] = data["topics"].fillna("").str.strip()
data = data.loc[data["text"].str.strip().ne("") & data["topics"].ne("")].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data["topics"])
X_train_val_text, X_test_text, y_train_val, y_test = train_test_split(
    data["text"], y, test_size=0.2, random_state=42, stratify=y
)
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_train_val_text, y_train_val, test_size=0.125,
    random_state=42, stratify=y_train_val
)

# TF-IDF di-fit pada data latih saja untuk mencegah kebocoran data.
vectorizer = TfidfVectorizer(max_features=512, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
X_val = vectorizer.transform(X_val_text).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_text).toarray().astype(np.float32)
num_classes = len(label_encoder.classes_)
class_ids = np.arange(num_classes)
weights = compute_class_weight(class_weight="balanced", classes=class_ids, y=y_train)
class_weights = dict(zip(class_ids, weights))

print("Jumlah data training:", len(X_train))
print("Jumlah data validasi:", len(X_val))
print("Jumlah data testing :", len(X_test))
print("Jumlah fitur TF-IDF :", X_train.shape[1])
print("Kelas topik:", list(label_encoder.classes_))

# ==========================================
# 2. MEMBANGUN MODEL JST
# ==========================================
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])
model.summary()

# ==========================================
# 3. KOMPILASI DAN PELATIHAN MODEL
# ==========================================
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    verbose=1
)

# ==========================================
# 4. EVALUASI MODEL
# ==========================================
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
test_predictions = np.argmax(model.predict(X_test, verbose=0), axis=1)
print("\n=== HASIL EVALUASI ===")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")
print("\n=== LAPORAN PER KELAS ===")
print(classification_report(
    y_test,
    test_predictions,
    labels=class_ids,
    target_names=label_encoder.classes_,
    zero_division=0
))


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import time
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# ==========================================
# 1. LOAD DAN SIAPKAN DATASET KOMDIGI
# ==========================================
data_candidates = [
    Path("data/tabular/komdigi_hoaks.csv"),
    Path("../../../data/tabular/komdigi_hoaks.csv"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("File data/tabular/komdigi_hoaks.csv tidak ditemukan.")

data = pd.read_csv(DATA_PATH, usecols=["title", "body_text", "topics"])
data["text"] = data["title"].fillna("") + " " + data["body_text"].fillna("")
data["topics"] = data["topics"].fillna("").str.strip()
data = data.loc[data["text"].str.strip().ne("") & data["topics"].ne("")].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data["topics"])
X_train_val_text, X_test_text, y_train_val, y_test = train_test_split(
    data["text"], y, test_size=0.2, random_state=42, stratify=y
)
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_train_val_text, y_train_val, test_size=0.125,
    random_state=42, stratify=y_train_val
)

# Fit TF-IDF pada data latih saja; validasi dan tes hanya ditransformasikan.
vectorizer = TfidfVectorizer(max_features=512, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
X_val = vectorizer.transform(X_val_text).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_text).toarray().astype(np.float32)
num_classes = len(label_encoder.classes_)
class_ids = np.arange(num_classes)
weights = compute_class_weight(class_weight="balanced", classes=class_ids, y=y_train)
class_weights = dict(zip(class_ids, weights))

print("Jumlah data:", len(data))
print("Jumlah fitur TF-IDF:", X_train.shape[1])
print("Kelas target:", list(label_encoder.classes_))

# ==========================================
# 2. KONFIGURASI EKSPERIMEN
# ==========================================
experiments = [
    {"nama": "Model Awal", "neuron": [128, 64], "aktivasi": "relu"},
    {"nama": "Neuron 256-128", "neuron": [256, 128], "aktivasi": "relu"},
    {"nama": "3 Hidden Layer", "neuron": [256, 128, 64], "aktivasi": "relu"},
    {"nama": "Aktivasi Sigmoid", "neuron": [128, 64], "aktivasi": "sigmoid"}
]

# ==========================================
# 3. PROSES EKSPERIMEN
# ==========================================
hasil = []

for exp in experiments:
    print("\n" + "=" * 50)
    print("Eksperimen:", exp["nama"])
    print("Neuron:", exp["neuron"])
    print("Aktivasi:", exp["aktivasi"])
    print("=" * 50)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    model = Sequential([Input(shape=(X_train.shape[1],))])
    for neuron in exp["neuron"]:
        model.add(Dense(neuron, activation=exp["aktivasi"]))
    model.add(Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    waktu_mulai = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=32,
        validation_data=(X_val, y_val),
        class_weight=class_weights,
        verbose=0
    )
    waktu_pelatihan = time.perf_counter() - waktu_mulai

    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    predictions = np.argmax(model.predict(X_test, verbose=0), axis=1)
    macro_f1 = f1_score(y_test, predictions, average="macro", zero_division=0)

    hasil.append({
        "Eksperimen": exp["nama"],
        "Neuron": str(exp["neuron"]),
        "Aktivasi": exp["aktivasi"],
        "Jumlah Parameter": model.count_params(),
        "Test Loss": test_loss,
        "Test Accuracy (%)": test_accuracy * 100,
        "Macro-F1": macro_f1,
        "Waktu Pelatihan (detik)": waktu_pelatihan,
        "Val Accuracy (%)": history.history["val_accuracy"][-1] * 100,
        "Prediksi": predictions
    })

    print(f"Test Loss       : {test_loss:.4f}")
    print(f"Test Accuracy   : {test_accuracy * 100:.2f}%")
    print(f"Test Macro-F1   : {macro_f1:.4f}")
    print(f"Waktu Pelatihan : {waktu_pelatihan:.2f} detik")

# ==========================================
# 4. PERBANDINGAN HASIL
# ==========================================
df_hasil = pd.DataFrame([
    {key: value for key, value in hasil_model.items() if key != "Prediksi"}
    for hasil_model in hasil
])
print("\n===== TABEL PERBANDINGAN EKSPERIMEN =====")
display(df_hasil.sort_values(by="Macro-F1", ascending=False).reset_index(drop=True).round(4))

# Macro-F1 digunakan untuk memilih model karena distribusi topik tidak seimbang.
terbaik = max(hasil, key=lambda hasil_model: hasil_model["Macro-F1"])
print("\n===== MODEL TERBAIK BERDASARKAN MACRO-F1 =====")
print("Eksperimen:", terbaik["Eksperimen"])
print("Akurasi:", f"{terbaik['Test Accuracy (%)']:.2f}%")
print("Macro-F1:", f"{terbaik['Macro-F1']:.4f}")
print(classification_report(
    y_test,
    terbaik["Prediksi"],
    labels=class_ids,
    target_names=label_encoder.classes_,
    zero_division=0
))


Eksperimen ini menggunakan dataset `komdigi_hoaks.csv` untuk mengklasifikasikan artikel berdasarkan kolom `topics`. Teks judul dan isi artikel diubah menjadi fitur TF-IDF, lalu digunakan oleh jaringan saraf tiruan untuk memprediksi enam topik.

Data dibagi secara stratifikasi menjadi data latih, validasi, dan tes. TF-IDF hanya di-fit pada data latih. Karena jumlah artikel tiap topik tidak seimbang, pelatihan menggunakan bobot kelas dan Macro-F1 digunakan untuk memilih konfigurasi terbaik. Jalankan ulang sel di atas untuk memperoleh metrik aktual; akurasi saja dapat menutupi performa rendah pada topik yang jumlah datanya sedikit.